# Image Generation pipeline

First, we force the download of the required imports.

In [ ]:
# === Install compatible library versions === #

%pip install -q ftfy==6.1.3
%pip install -q diffusers==0.30.2 transformers==4.46.3
%pip install -q torch torchvision safetensors accelerate hf_transfer numpy pillow python-dotenv

# === Verify versions installed === #

import importlib

libs = ["torch", "torchvision", "diffusers", "transformers",
        "accelerate", "safetensors", "ftfy", "numpy", "PIL"]

print("\nInstalled library versions:")

for lib in libs:
    
    try:
        module = importlib.import_module(lib if lib != "PIL" else "PIL.Image")
        version = getattr(module, "__version__", "built-in")
        print(f"  {lib:<15} → {version}")
        
    except Exception as e:
        print(f"  {lib:<15} {e}")

print("\nEnvironment setup complete and verified.")

Then, in this piece of code we load up, from our hugging face account, a stable diffusion model and its weights.

In [ ]:
import os
import torch
import warnings

from dotenv import load_dotenv
from huggingface_hub import login
from diffusers import StableDiffusionXLPipeline

# Remove warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

MODEL_ID = "Viennoiserie/Pony_V3"
login(token=hf_token)

pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID,
                                                 torch_dtype=torch.float16,        
                                                 use_safetensors=True).to("cuda")

print("Model loaded successfully with minimal warnings !")

Afterwards, in this piece of code, using the previous model, we generate an image with the prompt we entered.

In [ ]:
prompt1 = ""
prompt2 = ""

prompt = prompt1 + prompt2

negative_prompt = "blurry, low quality, distorted, extra limbs, unfocused eyes, misaligned pupils, cross-eyed, eyes looking at camera when not intended, unnatural gaze, bad anatomy"

image = pipe(prompt=prompt,
             negative_prompt=negative_prompt,

             num_inference_steps=50,
             height=1024,
             width=1024,
             
             guidance_scale=7.5).images[0]

image.save(f"Output/test.jpg")

Finally, we save it :)
